In [ ]:

%load_ext autoreload
%autoreload 2

import pandas as pd 
import matplotlib.pyplot as plt

import sys
sys.path.append("..")

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from src.evaluacion.backtesting import walk_forward
from src.features.feature_sets import get_feature_sets, TARGET


# 06 · Modelos simples: ¿hace falta un modelo complejo?

Compara cuatro modelos **en igualdad de condiciones**: mismos datos (46 meses, 97 features),
mismo walk-forward (34 meses evaluados) y misma métrica (MAE):
naive D-1, regresión lineal, árbol de decisión y XGBoost (receta de producción).


In [ ]:
df =pd.read_parquet("../data/processed/tabla_features.parquet").sort_values("datetime_utc")
df = df[df["entrenable"]]

In [ ]:
feats = get_feature_sets(df)["predictivo"]

# Controles de seguridad (anti-leakage y anti-NaN)
assert TARGET not in feats and "precio_portugal" not in feats
assert not any(c.endswith("_real") for c in feats)

df["meses"] = df["datetime_utc"].dt.to_period("M")

assert len(feats) == 97, f"se esperaban 97 features y hay {len(feats)}"
assert df[feats].isna().sum().sum() == 0, "hay NaN en las features"
print(f"{len(df)} filas | {df['meses'].nunique()} meses | {len(feats)} features")

In [ ]:
df_wf_naive = walk_forward(df, TARGET, lambda tr, te: te["precio_lag_24"])
print(f"Naive D-1 → MAE {df_wf_naive['MAE'].mean():.2f} ± {df_wf_naive['MAE'].std():.2f}")

In [ ]:
modelo_lineal = LinearRegression()
df_wf_lineal = walk_forward(
    df,TARGET,
    lambda tr, te: modelo_lineal.fit(tr[feats],tr[TARGET]).predict(te[feats]),
)
print(f"Regresion Lineal walk-forward → MAE {df_wf_lineal['MAE'].mean():.2f} ± {df_wf_lineal['MAE'].std():.2f}")

In [ ]:
resultado_arbol = {}
profundidades=[4, 6, 8, 10, 12]

for profundidad in profundidades:
    modelo_arbol =DecisionTreeRegressor(max_depth=profundidad,
                                        random_state=42)
    df_wf_arbol= walk_forward(
        df,TARGET,
        lambda tr,te : modelo_arbol.fit(tr[feats],tr[TARGET]).predict(te[feats]),
    )
    resultado_arbol[profundidad] =df_wf_arbol
    
    print(f"Árbol max_depth={profundidad} → MAE {df_wf_arbol['MAE'].mean():.2f} ± {df_wf_arbol['MAE'].std():.2f}")


In [ ]:
modelo_xgb = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

df_wf_xgb = walk_forward(
    df,TARGET,
    lambda tr, te: modelo_xgb.fit(tr[feats],tr[TARGET]).predict(te[feats]),
)
print(f"XGBoost walk-forward → MAE {df_wf_xgb['MAE'].mean():.2f} ± {df_wf_xgb['MAE'].std():.2f}")

In [ ]:
# Resultados de los cuatro modelos, todos sobre los mismos 34 meses
modelos = {
    "Naive D-1": df_wf_naive,
    "Regresión lineal": df_wf_lineal,
    "Árbol (max_depth=6)": resultado_arbol[6],
    "XGBoost": df_wf_xgb,
}

filas = []
for nombre, df_wf in modelos.items():
    filas.append({
        "modelo": nombre,
        "MAE": df_wf["MAE"].mean(),
        "desviación": df_wf["MAE"].std(),
        # Meses en que el modelo tiene menos error que el naive (comparación mes a mes)
        "meses que gana al naive": (df_wf["MAE"].values < df_wf_naive["MAE"].values).sum(),
    })

tabla_modelos = (
    pd.DataFrame(filas)
    .set_index("modelo")
    .sort_values("MAE")
    .round(2)
)
tabla_modelos["mejora vs naive (%)"] = (
    (tabla_modelos["MAE"] / df_wf_naive["MAE"].mean() - 1) * 100
).round(1)

display(tabla_modelos)

In [ ]:
tabla_modelos.to_csv("../reports/comparacion_modelos.csv")

## Conclusión

| Modelo | MAE (€/MWh) | Desviación | Meses que gana al naive (de 34) | vs naive |
|---|---|---|---|---|
| **XGBoost** | **17,21** | 4,78 | **27** | **−11,2 %** |
| Regresión lineal | 18,29 | 3,62 | 26 | −5,6 % |
| Naive D-1 | 19,38 | 6,15 | — | — |
| Árbol de decisión (max_depth = 6) | 20,03 | 5,21 | 23 | +3,4 % |

**XGBoost es el mejor modelo**: menor error medio y gana al naive en 27 de 34 meses.
La complejidad está justificada, pero por un margen **moderado**: frente a la regresión
lineal la mejora es de 1,08 €/MWh (≈ 6 %).

- **Regresión lineal**: mejora al naive y es el modelo **más estable** (menor desviación),
  pero no capta las relaciones no lineales del precio (horas a precio cero, picos).
  Sería una alternativa razonable si primara la simplicidad.
- **Árbol de decisión**: gana al naive en la mayoría de meses (23), pero su media es peor.
  Pierde por mucho en los cambios de régimen: solo puede predecir unos pocos valores
  (64 hojas con profundidad 6) y **no puede extrapolar** por encima de los precios que vio
  al entrenar. La profundidad muestra la forma de U típica: infraajuste con 4,
  sobreajuste con 12.
- **XGBoost** combina la capacidad no lineal de los árboles con la suma de 400 árboles
  que corrigen los errores de los anteriores, lo que da predicciones finas y robustas.

**Notas de rigor**

- Todos los modelos se evaluaron con los mismos datos, el mismo walk-forward y la misma
  métrica. Estas cifras no son comparables con las de septiembre (16,49 / 18,61),
  que tenían un mes menos de datos.
- La profundidad del árbol se eligió mirando los mismos meses con los que se mide:
  la cifra del árbol es ligeramente optimista.
- **LSTM**: no se recalculó (entrenada en Colab). En septiembre dio 16,59 ± 7,96, cuando
  XGBoost daba 15,70 ± 5,29, sobre otro periodo; aquella comparación favorecía a XGBoost,
  que aún tenía leakage.
- **Trabajo futuro**: comparar los modelos también en **regret** (valor para la batería),
  no solo en MAE.